In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install gradio -q

In [ ]:
import torch
import torchvision.models as models
import torch.nn as nn

from torchvision import transforms
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import json
import gradio as gr
import cv2
import numpy as np


In [ ]:
def predict_xray(input_image):

    # =========================================
    # IMAGE PREPROCESSING
    # =========================================

    image = input_image.convert("RGB")

    image_tensor = transform(image).unsqueeze(0).to(device)

    # =========================================
    # MODEL INFERENCE
    # =========================================

    with torch.no_grad():

        outputs = model(image_tensor)

        probs = torch.sigmoid(outputs).cpu().numpy()[0]

    # =========================================
    # CREATE RESULTS TABLE
    # =========================================

    results = []

    for label, prob, threshold in zip(
        ALL_LABELS,
        probs,
        FINAL_THRESHOLDS
    ):

        prediction = prob > threshold

        results.append({
            "Disease": label,
            "Probability": round(float(prob), 3),
            "Threshold": threshold,
            "Prediction": (
                "Positive"
                if prediction
                else "Negative"
            )
        })

    df = pd.DataFrame(results)

    # حذف القيم الضعيفة
    df = df[df["Probability"] >= 0.10]

    # ترتيب النتائج
    df = df.sort_values(
        by="Probability",
        ascending=False
    )

    # =========================================
    # AI SUMMARY
    # =========================================

    top_result = df.iloc[0]

    summary = (
        f"Top Prediction: "
        f"{top_result['Disease']} "
        f"({top_result['Probability']*100:.1f}%)"
    )

    return df, summary

In [ ]:
def create_probability_plot(df):

    plt.close('all')

    fig = plt.figure(figsize=(12,5))

    plt.bar(
        df["Disease"],
        df["Probability"]
    )

    plt.xticks(rotation=45)

    plt.ylabel("Probability")

    plt.xlabel("Disease")

    plt.title(
        "Disease Prediction Probabilities"
    )

    plt.tight_layout()

    return fig

In [ ]:
def create_gradcam(input_image):

    image = input_image.convert("RGB")

    image_tensor = transform(image).unsqueeze(0).to(device)

    activations = None
    gradients = None

    # حفظ activations
    def save_activation(module, input, output):

        nonlocal activations

        activations = output

    # حفظ gradients
    def save_gradient(grad):

        nonlocal gradients

        gradients = grad

    target_layer = model.features[-1]

    hook_handle = target_layer.register_forward_hook(
        save_activation
    )

    model.zero_grad()

    output = model(image_tensor)

    activations.register_hook(save_gradient)

    pred_index = torch.argmax(output)

    output[0, pred_index].backward()

    pooled_gradients = torch.mean(
        gradients,
        dim=[0, 2, 3]
    )

    activation = activations.detach().clone()

    for i in range(activation.shape[1]):

        activation[:, i, :, :] *= pooled_gradients[i]

    heatmap = torch.mean(
        activation,
        dim=1
    ).squeeze()

    heatmap = torch.relu(heatmap)

    heatmap /= torch.max(heatmap)

    heatmap = heatmap.cpu().numpy()

    hook_handle.remove()

    # تجهيز الصورة
    img = np.array(image)

    img = cv2.resize(
        img,
        (224,224)
    )

    heatmap = cv2.resize(
        heatmap,
        (224,224)
    )

    heatmap_colored = cv2.applyColorMap(
        np.uint8(255 * heatmap),
        cv2.COLORMAP_JET
    )

    superimposed_img = cv2.addWeighted(
        img,
        0.6,
        heatmap_colored,
        0.4,
        0
    )

    return superimposed_img

In [ ]:
# =========================================
# GRAD-CAM BOUNDING BOX - STABLE VERSION
# =========================================

def create_gradcam_box(input_image):

    image = input_image.convert("RGB")

    image_tensor = transform(
        image
    ).unsqueeze(0).to(device)

    activations = None
    gradients = None

    # =====================================
    # SAVE ACTIVATIONS
    # =====================================

    def save_activation(module, input, output):

        nonlocal activations

        activations = output

    # =====================================
    # SAVE GRADIENTS
    # =====================================

    def save_gradient(grad):

        nonlocal gradients

        gradients = grad

    target_layer = model.features[-1]

    hook_handle = target_layer.register_forward_hook(
        save_activation
    )

    try:

        # =====================================
        # FORWARD PASS
        # =====================================

        model.zero_grad()

        output = model(image_tensor)

        # التأكد من تسجيل activations
        if activations is None:

            raise RuntimeError(
                "Grad-CAM failed: activations were not captured."
            )

        # =====================================
        # SELECT PREDICTION
        # =====================================

        pred_index = torch.argmax(output)

        # تسجيل gradient BEFORE backward
        activations.register_hook(
            save_gradient
        )

        # =====================================
        # BACKWARD PASS
        # =====================================

        output[0, pred_index].backward()

        # التأكد من الحصول على gradients
        if gradients is None:

            raise RuntimeError(
                "Grad-CAM failed: gradients were not captured."
            )

        # =====================================
        # CALCULATE GRAD-CAM
        # =====================================

        pooled_gradients = torch.mean(
            gradients,
            dim=[0, 2, 3]
        )

        activation = (
            activations
            .detach()
            .clone()
        )

        for i in range(
            activation.shape[1]
        ):

            activation[:, i, :, :] *= (
                pooled_gradients[i]
            )

        heatmap = torch.mean(
            activation,
            dim=1
        ).squeeze()

        heatmap = torch.relu(
            heatmap
        )

        # =====================================
        # NORMALIZE HEATMAP
        # =====================================

        max_value = torch.max(
            heatmap
        )

        if max_value > 0:

            heatmap = (
                heatmap / max_value
            )

        heatmap = (
            heatmap
            .detach()
            .cpu()
            .numpy()
        )

    finally:

        # إزالة hook دائماً
        hook_handle.remove()

        model.zero_grad()

    # =====================================
    # PREPARE IMAGE
    # =====================================

    img = np.array(
        image
    )

    img = cv2.resize(
        img,
        (224, 224)
    )

    heatmap = cv2.resize(
        heatmap,
        (224, 224)
    )

    # =====================================
    # CREATE ACTIVATION MASK
    # =====================================

    threshold = 0.60

    mask = np.uint8(
        heatmap >= threshold
    ) * 255

    # =====================================
    # CLEAN MASK
    # =====================================

    kernel = np.ones(
        (5, 5),
        np.uint8
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    # =====================================
    # FIND CONTOURS
    # =====================================

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    result = img.copy()

    # =====================================
    # DRAW BOUNDING BOX
    # =====================================

    if contours:

        # أكبر منطقة نشطة
        largest_contour = max(
            contours,
            key=cv2.contourArea
        )

        area = cv2.contourArea(
            largest_contour
        )

        # تجاهل المناطق الصغيرة جداً
        if area > 100:

            x, y, w, h = cv2.boundingRect(
                largest_contour
            )

            # رسم المستطيل
            cv2.rectangle(
                result,
                (x, y),
                (x + w, y + h),
                (255, 0, 0),
                2
            )

            # كتابة التوضيح
            cv2.putText(
                result,
                "AI Attention Region",
                (x, max(y - 8, 15)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 0, 0),
                1,
                cv2.LINE_AA
            )

    return result

In [ ]:
# =========================================
# DISEASE INFORMATION DATABASE
# =========================================

DISEASE_INFO = {

    "Atelectasis": {
        "en": "Partial collapse of the lung causing reduced air volume.",
        "ar": "انخماص الرئة: انهيار جزئي في الرئة يؤدي إلى نقص حجم الهواء."
    },

    "Cardiomegaly": {
        "en": "Enlargement of the heart which may indicate cardiac disease.",
        "ar": "تضخم القلب: زيادة حجم القلب وقد يدل على أمراض قلبية."
    },

    "Effusion": {
        "en": "Fluid accumulation around the lungs.",
        "ar": "الانصباب الجنبي: تجمع السوائل حول الرئتين."
    },

    "Infiltration": {
        "en": "Substance accumulation in lung tissue often related to infection.",
        "ar": "الارتشاح الرئوي: تراكم مواد داخل أنسجة الرئة وغالباً يرتبط بالالتهاب."
    },

    "Mass": {
        "en": "Abnormal tissue growth in the chest.",
        "ar": "كتلة: نمو غير طبيعي للأنسجة داخل الصدر."
    },

    "Nodule": {
        "en": "Small abnormal rounded growth in the lung.",
        "ar": "عقيدة رئوية: نمو صغير دائري غير طبيعي في الرئة."
    },

    "Pneumonia": {
        "en": "Lung infection causing inflammation of air sacs.",
        "ar": "الالتهاب الرئوي: عدوى تسبب التهاب الحويصلات الهوائية."
    },

    "Pneumothorax": {
        "en": "Collapsed lung caused by air leakage into the pleural space.",
        "ar": "استرواح الصدر: انهيار الرئة بسبب تسرب الهواء."
    },

    "Consolidation": {
        "en": "Region of lung filled with fluid instead of air.",
        "ar": "التكثف الرئوي: امتلاء جزء من الرئة بالسوائل بدلاً من الهواء."
    },

    "Edema": {
        "en": "Fluid accumulation inside the lungs.",
        "ar": "الوذمة الرئوية: تجمع السوائل داخل الرئة."
    },

    "Emphysema": {
        "en": "Chronic lung disease damaging air sacs.",
        "ar": "النفاخ الرئوي: مرض مزمن يسبب تلف الحويصلات الهوائية."
    },

    "Fibrosis": {
        "en": "Scarring and thickening of lung tissue.",
        "ar": "التليف الرئوي: تندب وسماكة في أنسجة الرئة."
    },

    "Pleural_Thickening": {
        "en": "Thickening of the pleural lining surrounding the lungs.",
        "ar": "سماكة غشاء الجنب المحيط بالرئتين."
    },

    "Hernia": {
        "en": "Abnormal protrusion of internal tissue or organ.",
        "ar": "الفتق: بروز غير طبيعي لعضو أو نسيج داخلي."
    },

    "No Finding": {
        "en": "No visible abnormality detected.",
        "ar": "لا توجد نتائج مرضية ظاهرة."
    }
}

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [ ]:
with open("/content/drive/MyDrive/final_labels.json", "r") as f:
    ALL_LABELS = json.load(f)

with open("/content/drive/MyDrive/final_thresholds.json", "r") as f:
    FINAL_THRESHOLDS = json.load(f)

print(ALL_LABELS)
print(FINAL_THRESHOLDS)

['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia', 'No Finding']
[0.40000000000000013, 0.5000000000000001, 0.5000000000000001, 0.3500000000000001, 0.3500000000000001, 0.3500000000000001, 0.30000000000000004, 0.5000000000000001, 0.30000000000000004, 0.6500000000000001, 0.45000000000000007, 0.45000000000000007, 0.3500000000000001, 0.8500000000000002, 0.1]


In [ ]:
# NEW THRESHOLDS

FINAL_THRESHOLDS = [
    0.35,  # Atelectasis
    0.35,  # Cardiomegaly
    0.45,  # Effusion
    0.40,  # Infiltration
    0.30,  # Mass
    0.40,  # Nodule
    0.30,  # Pneumonia
    0.45,  # Pneumothorax
    0.35,  # Consolidation
    0.40,  # Edema
    0.35,  # Emphysema
    0.30,  # Fibrosis
    0.25,  # Pleural_Thickening
    0.40,  # Hernia
    0.50   # No Finding
]

print("✅ New thresholds loaded")
print(FINAL_THRESHOLDS)

✅ New thresholds loaded
[0.35, 0.35, 0.45, 0.4, 0.3, 0.4, 0.3, 0.45, 0.35, 0.4, 0.35, 0.3, 0.25, 0.4, 0.5]


In [ ]:
model = models.densenet121(
    weights=models.DenseNet121_Weights.DEFAULT
)

num_features = model.classifier.in_features

model.classifier = nn.Linear(num_features, 15)

model = model.to(device)

In [ ]:
checkpoint = torch.load(
    "/content/drive/MyDrive/best_model.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state"])

model.eval()

print("✅ Best model loaded")

✅ Best model loaded


In [ ]:
# تحميل أوزان النموذج
model.load_state_dict(checkpoint["model_state"])

# استخراج المعلومات
ALL_LABELS = checkpoint.get("labels", ALL_LABELS)

best_f1 = checkpoint.get("best_f1", None)
auc = checkpoint.get("auc", None)
model_name = checkpoint.get("model_name", "Unknown")

model.eval()

print("✅ Model loaded successfully")

✅ Model loaded successfully


In [ ]:
print("="*50)
print("MODEL DASHBOARD")
print("="*50)

print(f"Model name: {model_name}")
print(f"Best F1: {best_f1}")
print(f"AUC: {auc}")

print("\nNumber of labels:", len(ALL_LABELS))
print("Number of thresholds:", len(FINAL_THRESHOLDS))

print("\nClassifier layer:")
print(model.classifier)

print("\nOutput size:", model.classifier.out_features)

MODEL DASHBOARD
Model name: DenseNet121
Best F1: 0.35971619608409094
AUC: 0.8214181236156025

Number of labels: 15
Number of thresholds: 15

Classifier layer:
Linear(in_features=1024, out_features=15, bias=True)

Output size: 15


In [ ]:
assert len(ALL_LABELS) == len(FINAL_THRESHOLDS), \
"Mismatch between labels and thresholds!"

print("✅ Labels and thresholds are aligned")

✅ Labels and thresholds are aligned


In [ ]:
print("="*50)
print("MODEL INFORMATION")
print("="*50)

# عدد الـ labels
print("\nNumber of labels:")
print(len(ALL_LABELS))

# أسماء الـ labels
print("\nLabels:")
print(ALL_LABELS)

# عدد الـ thresholds الحالية
print("\nNumber of thresholds:")
print(len(FINAL_THRESHOLDS))

# قيم الـ thresholds الحالية
print("\nCurrent thresholds:")
print(FINAL_THRESHOLDS)

# شكل طبقة الخرج
print("\nModel classifier:")
print(model.classifier)

# عدد الـ outputs
print("\nOutput features:")
print(model.classifier.out_features)

print("\n" + "="*50)
print("CHECKING AVAILABLE VARIABLES")
print("="*50)

# فحص المتغيرات الموجودة
available_vars = list(globals().keys())

important_vars = [
    "train_loader",
    "val_loader",
    "test_loader",
    "train_df",
    "val_df",
    "test_df",
    "dataset",
    "labels",
    "targets"
]

for var in important_vars:
    if var in available_vars:
        print(f"FOUND: {var}")
    else:
        print(f"NOT FOUND: {var}")

MODEL INFORMATION

Number of labels:
15

Labels:
['Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia', 'No Finding']

Number of thresholds:
15

Current thresholds:
[0.35, 0.35, 0.45, 0.4, 0.3, 0.4, 0.3, 0.45, 0.35, 0.4, 0.35, 0.3, 0.25, 0.4, 0.5]

Model classifier:
Linear(in_features=1024, out_features=15, bias=True)

Output features:
15

CHECKING AVAILABLE VARIABLES
NOT FOUND: train_loader
NOT FOUND: val_loader
NOT FOUND: test_loader
NOT FOUND: train_df
NOT FOUND: val_df
NOT FOUND: test_df
NOT FOUND: dataset
NOT FOUND: labels
NOT FOUND: targets


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [ ]:
# =========================================
# ADVANCED CLINICAL REPORT
# =========================================

def generate_clinical_report(df):

    positive_df = df[
        df["Prediction"] == "Positive"
    ]

    # =====================================
    # NO FINDINGS
    # =====================================

    if len(positive_df) == 0:

        report = """
CHEST X-RAY REPORT
========================================

ENGLISH REPORT:

No significant abnormality detected.

IMPRESSION:
No acute cardiopulmonary process.

----------------------------------------

التقرير العربي:

لا توجد شذوذات واضحة في صورة الأشعة.

الانطباع:
لا توجد مشكلة قلبية أو رئوية حادة واضحة.

========================================
AI GENERATED REPORT
NOT FOR CLINICAL USE
"""

        return report

    # =====================================
    # POSITIVE FINDINGS
    # =====================================

    report = """
CHEST X-RAY REPORT
========================================
"""

    for _, row in positive_df.iterrows():

        disease = row["Disease"]

        probability = row["Probability"] * 100

        info = DISEASE_INFO.get(
            disease,
            {
                "en": "No description available.",
                "ar": "لا يوجد وصف متوفر."
            }
        )

        report += f"""

Disease: {disease}

Probability: {probability:.1f}%

EN:
{info['en']}

AR:
{info['ar']}

----------------------------------------
"""

    # =====================================
    # IMPRESSION
    # =====================================

    top_disease = positive_df.iloc[0]["Disease"]

    report += f"""

IMPRESSION:
Findings suspicious for {top_disease}.

الانطباع:
النتائج تشير إلى احتمال وجود {top_disease}.

========================================
AI GENERATED REPORT
NOT FOR CLINICAL USE
"""

    return report

In [ ]:
# =========================================
# FULL PIPELINE
# =========================================

def full_prediction_pipeline(input_image):

    # =====================================
    # PREDICTIONS
    # =====================================

    df, summary = predict_xray(input_image)

    # =====================================
    # PROBABILITY PLOT
    # =====================================

    plot = create_probability_plot(df)

    # =====================================
    # GRAD-CAM
    # =====================================

    gradcam = create_gradcam(input_image)

    # =====================================
    # GRAD-CAM BOUNDING BOX
    # =====================================

    boxed_image = create_gradcam_box(input_image)

    # =====================================
    # CLINICAL REPORT
    # =====================================

    report = generate_clinical_report(df)

    # =====================================
    # RETURN ALL OUTPUTS
    # =====================================

    return (
        df,
        plot,
        gradcam,
        boxed_image,
        summary,
        report
    )

In [ ]:
# =========================================
# ADVANCED GRADIO DASHBOARD
# =========================================
with gr.Blocks(theme=gr.themes.Soft()) as interface:

    gr.Markdown(
        "# Chest X-ray AI Diagnostic System"
    )

    gr.Markdown(
        "DenseNet121 + Grad-CAM + Clinical Report"
    )

    with gr.Row():

        input_image = gr.Image(
            type="pil",
            label="Upload Chest X-ray"
        )

    analyze_btn = gr.Button(
        "Analyze X-ray"
    )

    with gr.Tabs():

        # =====================================
        # TAB 1
        # =====================================

        with gr.Tab("Prediction Results"):

            results_output = gr.Dataframe(
                label="Prediction Results"
            )

            summary_output = gr.Textbox(
                label="AI Summary"
            )

        # =====================================
        # TAB 2
        # =====================================

        with gr.Tab("Visualization"):

            plot_output = gr.Plot(
                label="Probability Chart"
            )

            gradcam_output = gr.Image(
                label="Grad-CAM Heatmap"
            )

            boxed_output = gr.Image(
                label="AI Attention Region"
            )

        # =====================================
        # TAB 3
        # =====================================

        with gr.Tab("Clinical Report"):

            report_output = gr.Textbox(
                label="Clinical Report",
                lines=20
            )

    # =========================================
    # BUTTON ACTION
    # =========================================

    analyze_btn.click(

        fn=full_prediction_pipeline,

        inputs=input_image,

        outputs=[
            results_output,
            plot_output,
            gradcam_output,
            boxed_output,
            summary_output,
            report_output
        ]
    )

/tmp/ipykernel_16032/764932458.py:4: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as interface:


In [ ]:
gr.close_all()

interface.launch(
    debug=True,
    share=True
)